In [ ]:
%cd /kaggle/working
!git clone https://github.com/comfyanonymous/ComfyUI
!git clone https://github.com/OneHome/comfy-inference-server
!pip install -q -r ComfyUI/requirements.txt
!pip install -q -r comfy-inference-server/requirements.txt

In [ ]:
''' Definir el comando %download (ejecutar una vez al inicio) '''
import os, re, shlex, subprocess
from IPython.core.magic import register_line_magic

@register_line_magic
def download(line):
    parts = shlex.split(line)
    url = parts[0]
    name = parts[1] if len(parts) > 1 else None

    # Convierte enlaces de página de Civitai a enlace directo de descarga
    if "civitai.com" in url and "/api/download/" not in url:
        m = re.search(r"modelVersionId=(\d+)", url)
        if m:
            url = f"https://civitai.com/api/download/models/{m.group(1)}"
        else:
            print("⚠️ Falta modelVersionId en la URL de Civitai; usa la URL con ?modelVersionId=...")
            return

    # Token opcional de Civitai (os.environ["CIVITAI_TOKEN"] = "tu_token")
    token = os.environ.get("CIVITAI_TOKEN")
    if token and "civitai.com" in url:
        url += ("&" if "?" in url else "?") + f"token={token}"

    cmd = ["curl", "-L", "-C", "-", "--fail", "--progress-bar"]
    cmd += ["-o", name] if name else ["-OJ"]
    cmd.append(url)
    print(f"⬇️  {name or url}")
    subprocess.run(cmd)

In [ ]:
''' Diffusion models (almacenamiento temporal) '''
!mkdir -p /kaggle/temp/diffusion_models
!mkdir -p /kaggle/working/ComfyUI/models
!rmdir /kaggle/working/ComfyUI/models/diffusion_models 2>/dev/null
!ln -sfn /kaggle/temp/diffusion_models /kaggle/working/ComfyUI/models/diffusion_models

%cd -q /kaggle/temp/diffusion_models
%download https://civitai.com/models/2544636/wai-anima?modelVersionId=2859702 waiANIMA_v10.safetensors
%download https://civitai.com/api/download/models/2167369?type=Model&format=SafeTensor&size=pruned&fp=fp16&token=1acecff043206d9b18742585ea3fe5d5 waiNSFWIllustrious_v150.safetensors


In [ ]:
''' Text encoders (almacenamiento temporal) '''
!mkdir -p /kaggle/temp/text_encoders
!mkdir -p /kaggle/working/ComfyUI/models
!rmdir /kaggle/working/ComfyUI/models/text_encoders 2>/dev/null
!ln -sfn /kaggle/temp/text_encoders /kaggle/working/ComfyUI/models/text_encoders

%cd -q /kaggle/temp/text_encoders
%download https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors qwen_3_06b_base.safetensors

In [ ]:
''' VAEs (almacenamiento temporal) '''
!mkdir -p /kaggle/temp/vae
!mkdir -p /kaggle/working/ComfyUI/models
!rmdir /kaggle/working/ComfyUI/models/vae 2>/dev/null
!ln -sfn /kaggle/temp/vae /kaggle/working/ComfyUI/models/vae

%cd -q /kaggle/temp/vae
%download https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors qwen_image_vae.safetensors

In [ ]:
''' ControlNet (almacenamiento temporal) '''
!mkdir -p /kaggle/temp/controlnet
!mkdir -p /kaggle/working/ComfyUI/models
!rm -rf /kaggle/working/ComfyUI/models/controlnet
!ln -sfn /kaggle/temp/controlnet /kaggle/working/ComfyUI/models/controlnet

%cd -q /kaggle/temp/controlnet
%download https://huggingface.co/toyxyz/Anima_ControlNet-LLLite/resolve/main/anima_regional_4335.safetensors?download=true Regional.safetensors
%download https://huggingface.co/kohya-ss/Anima-LLLite/resolve/main/anima-lllite-any-test-like-v2.safetensors AnyTest.safetensors
%download https://huggingface.co/toyxyz/Anima_ControlNet-LLLite/resolve/main/anima_3dpose_6200.safetensors?download=true 3DPose.safetensors
%download https://huggingface.co/Cumot/JF/resolve/main/mistoLine_fp16.safetensors?download=true AnyLine.safetensors
%download https://huggingface.co/toyxyz/Anima_ControlNet-LLLite/resolve/main/anima_3dpose_outline_7000.safetensors?download=true Outline.safetensors
%download https://huggingface.co/kohya-ss/Anima-LLLite/resolve/main/anima-lllite-scribble-1.safetensors?download=true Scribble.safetensors
%download https://huggingface.co/Cumot/JF/resolve/main/animaTileRepair_v10.safetensors?download=true RepairV1.safetensors
%download https://huggingface.co/Cumot/JF/resolve/main/animaTileRepair_v20.safetensors?download=true RepairV2.safetensors
%download https://huggingface.co/buckets/Cumot/controlnet-openpose-illustrious-xl-bucket/resolve/diffusion_pytorch_model.safetensors?download=true PoseIllustrious.safetensors

In [ ]:
# LORAs (almacenamiento temporal)
!mkdir -p /kaggle/temp/loras
!mkdir -p /kaggle/working/ComfyUI/models
!rmdir /kaggle/working/ComfyUI/models/loras 2>/dev/null
!ln -sfn /kaggle/temp/loras /kaggle/working/ComfyUI/models/loras

%cd -q /kaggle/temp/loras

# Descargas de LoRAs usando wget seguro
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Kishimoto.safetensors" -O Kishimoto.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/FullFull.safetensors" -O FullFull.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Blueu.safetensors" -O Blueu.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Loud_House_Style.safetensors" -O Loud_House_Style.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Dogma.safetensors" -O Dogma.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Beebz.safetensors" -O Beebz.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/FULLH10.safetensors" -O FULLH10.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/nuwasaill.safetensors" -O nuwasaill.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/IllustriousNWStyleV1.safetensors" -O IllustriousNWStyleV1.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/NTR_MIX_4.0_LORA-000007.safetensors" -O NTR_MIX_4.0_LORA-000007.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/YellowElephantStyle.safetensors" -O YellowElephantStyle.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Raikage%20IL%20v1.1.safetensors" -O Raikage_IL.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Dogm%2010.safetensors" -O Dogm_10.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Simon_10.safetensors" -O Simon.safetensors
!wget -c "https://civitai.com/api/download/models/1820705" -O dmd2_sdxl_4step_lora.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/NARUTO_illustriousXL.safetensors" -O NARUTO_illustriousXL.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/anatomy_helper.safetensors" -O anatomy_helper.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/FH%2009.safetensors" -O FH_09.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/FH.safetensors" -O FH.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/illustriousXLv01_stabilizer_v1.198.safetensors" -O illustriousXLv01_stabilizer_v1.198.safetensors
!wget -c "https://huggingface.co/tianweiy/DMD2/resolve/main/dmd2_sdxl_4step_lora.safetensors" -O dmd2_sdxl_4step_lora.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Cosmos-Predict2.5-2B-base-distilled-LoRA.safetensors" -O Cosmos-Predict2.5-2B-base-distilled-LoRA.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/3D_animated_realistic_style.safetensors" -O 3D_animated_realistic_style.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/anima_edit_yaae_v02.safetensors" -O anima_edit_yaae_v02.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/lllsuperlllCGI_v2.safetensors" -O lllsuperlllCGI_v2.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/AnimeEditV2.safetensors" -O AnimeEditV2.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/lora_edit_ZeroTwo.safetensors" -O lora_edit_ZeroTwo.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/%40photorealisticV2edit-000055.safetensors" -O photorealisticV2edit-000055.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/anima-turbo-lora-v0.2.safetensors" -O anima-turbo-lora-v0.2.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Nartoon%2003.safetensors" -O Nartoon_03.safetensors
!wget -c "https://huggingface.co/Cumot/JF/resolve/main/Nartoon%2001.safetensors" -O Nartoon_01.safetensors

In [ ]:
%cd /kaggle/working/ComfyUI
!pip install -r requirements.txt

In [ ]:
!pip install -q gradio

import subprocess, secrets, inspect, time
from gradio import networking

PORT = 28888
PASSWORD = "redka"   # cámbiala

# 1) Arrancar el servidor en segundo plano
server = subprocess.Popen(
    ["python", "server.py",
     "--comfyui", "/kaggle/working/ComfyUI",
     "--bind", f"127.0.0.1:{PORT}",
     "--password", PASSWORD],
    cwd="/kaggle/working/comfy-inference-server",
)
time.sleep(10)  # darle tiempo a cargar ComfyUI

# 2) Abrir el túnel de Gradio hacia ese puerto
params = inspect.signature(networking.setup_tunnel).parameters
kw = dict(
    local_host="127.0.0.1",
    local_port=PORT,
    share_token=secrets.token_urlsafe(32),
    share_server_address=None,
)
if "share_server_tls_certificate" in params:   # versiones nuevas de Gradio
    kw["share_server_tls_certificate"] = None

url = networking.setup_tunnel(**kw)
print("HTTP :", url)
print("WS   :", url.replace("https://", "wss://"))

# 3) Mantener la celda viva
try:
    server.wait()
except KeyboardInterrupt:
    server.terminate()